# Cornell Chronicle: scrape article content

Reads `./Data/Cornell Articles Links.csv` and extracts date, author, source, headline, summary, body, and tags from every story page (JSON-LD first, HTML fallbacks), with retries, throttling, checkpoints, and resume support.

Output: `./Data/Cornell Articles.csv`


In [ ]:
from pathlib import Path
from time import sleep
import html
import json
import random
import re

import pandas as pd
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from tqdm.auto import tqdm
from urllib3.util.retry import Retry

In [ ]:
# -----------------------------
# Configuration
# -----------------------------
DATA_DIR = Path("./Data")
LINKS_PATH = DATA_DIR / "Cornell Articles Links.csv"
OUTPUT_PATH = DATA_DIR / "Cornell Articles.csv"

REQUEST_TIMEOUT = 30
DELAY_RANGE = (0.8, 1.5)
CHECKPOINT_EVERY = 10

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/126.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}

DATA_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
links = pd.read_csv(LINKS_PATH)

# Remove an index column produced by older versions of pandas code, if present.
links = links.loc[:, ~links.columns.str.startswith("Unnamed:")]

required_columns = {"Link"}
missing_columns = required_columns - set(links.columns)
if missing_columns:
    raise ValueError(f"Missing required columns in {LINKS_PATH}: {sorted(missing_columns)}")

links = links.drop_duplicates(subset="Link").reset_index(drop=True)
print(f"Loaded {len(links)} unique article links.")

In [ ]:
session = requests.Session()
session.headers.update(HEADERS)

retry_strategy = Retry(
    total=4,
    connect=4,
    read=4,
    status=4,
    backoff_factor=1.0,
    status_forcelist=(429, 500, 502, 503, 504),
    allowed_methods=frozenset(["GET"]),
    raise_on_status=False,
)

adapter = HTTPAdapter(max_retries=retry_strategy)
session.mount("https://", adapter)
session.mount("http://", adapter)

In [ ]:
def clean_text(value):
    """Decode HTML entities and collapse repeated whitespace."""
    if value is None:
        return ""
    return " ".join(html.unescape(str(value)).split())


def first_text(soup_or_tag, selectors):
    """Return clean text from the first matching CSS selector."""
    for selector in selectors:
        node = soup_or_tag.select_one(selector)
        if node:
            text = clean_text(node.get_text(" ", strip=True))
            if text:
                return text
    return ""


def first_attribute(soup_or_tag, selector_attribute_pairs):
    """Return the first non-empty attribute from a list of selectors."""
    for selector, attribute in selector_attribute_pairs:
        node = soup_or_tag.select_one(selector)
        if node:
            value = clean_text(node.get(attribute, ""))
            if value:
                return value
    return ""


def iter_json_objects(value):
    """Recursively yield dictionaries from JSON-LD structures."""
    if isinstance(value, dict):
        yield value
        for child in value.values():
            yield from iter_json_objects(child)
    elif isinstance(value, list):
        for child in value:
            yield from iter_json_objects(child)


def get_article_json_ld(soup):
    """Find a NewsArticle/Article object in JSON-LD, if available."""
    accepted_types = {
        "Article",
        "NewsArticle",
        "ReportageNewsArticle",
        "ScholarlyArticle",
    }

    for script in soup.select('script[type="application/ld+json"]'):
        raw = script.string or script.get_text()
        if not raw.strip():
            continue

        try:
            data = json.loads(raw)
        except (json.JSONDecodeError, TypeError):
            continue

        for obj in iter_json_objects(data):
            obj_type = obj.get("@type", "")
            types = set(obj_type if isinstance(obj_type, list) else [obj_type])
            if types & accepted_types:
                return obj

    return {}


def names_from_json_ld(value):
    """Convert JSON-LD person/organization objects into a readable string."""
    if not value:
        return ""

    if isinstance(value, str):
        return clean_text(value)

    if isinstance(value, dict):
        return clean_text(value.get("name", ""))

    if isinstance(value, list):
        names = [names_from_json_ld(item) for item in value]
        return ", ".join(name for name in names if name)

    return ""


SOURCE_SUFFIXES = (
    "Cornell Chronicle",
    "Cornell Tech",
    "Weill Cornell Medicine",
    "Cornell University",
)


def split_author_and_source(raw_byline):
    """Separate a recognized source suffix from a visible Cornell byline."""
    byline = re.sub(r"^\s*By\s+", "", clean_text(raw_byline), flags=re.IGNORECASE)

    for suffix in SOURCE_SUFFIXES:
        marker = f", {suffix}"
        if byline.endswith(marker):
            return byline[: -len(marker)].strip(), suffix
    return byline, ""

In [ ]:
def extract_body(soup, article_json):
    """Extract ordered body paragraphs and section headings.

    The first selectors target Cornell's Drupal body field. Later selectors are
    fallbacks for possible template changes.
    """
    body_selectors = [
        "article .field--name-body",
        "main .field--name-body",
        ".field--name-body",
        "article .field--type-text-with-summary",
        "main .story-body",
        "article .story-body",
        "main article",
        "article",
    ]

    container = None
    for selector in body_selectors:
        container = soup.select_one(selector)
        if container:
            break

    if container:
        # Work on a copy so removals do not affect other metadata extraction.
        body_soup = BeautifulSoup(str(container), "html.parser")

        unwanted_selectors = [
            "script",
            "style",
            "noscript",
            "nav",
            "footer",
            "form",
            "aside",
            "figure",
            ".caption",
            ".credit",
            ".byline",
            ".author",
            ".date",
            ".tags",
            ".share",
            ".social",
            ".related",
            ".media-inquiries",
            ".newsletter",
            ".subscribe",
        ]

        for selector in unwanted_selectors:
            for node in body_soup.select(selector):
                node.decompose()

        stop_markers = {
            "media inquiries",
            "media contact",
            "you might also like",
            "get cornell news delivered right to your inbox.",
        }

        blocks = []
        for node in body_soup.find_all(["p", "h2", "h3"]):
            text = clean_text(node.get_text(" ", strip=True))
            if not text:
                continue
            if text.casefold() in stop_markers:
                break
            if not blocks or text != blocks[-1]:
                blocks.append(text)

        if blocks:
            return "\n\n".join(blocks)

    # JSON-LD fallback.
    return clean_text(article_json.get("articleBody", ""))


def extract_tags(soup, article_json):
    """Extract topic/category labels without pulling unrelated navigation links."""
    selectors = [
        "article .field--name-field-topics a",
        "article .field--name-field-tags a",
        "main .field--name-field-topics a",
        "main .field--name-field-tags a",
        "article a[href*='/categories/']",
    ]

    tags = []
    for selector in selectors:
        for node in soup.select(selector):
            tag = clean_text(node.get_text(" ", strip=True))
            if tag and tag not in tags:
                tags.append(tag)

    if not tags:
        json_tags = article_json.get("keywords", [])
        if isinstance(json_tags, str):
            tags = [clean_text(tag) for tag in json_tags.split(",") if clean_text(tag)]
        elif isinstance(json_tags, list):
            tags = [clean_text(tag) for tag in json_tags if clean_text(tag)]

    return "; ".join(tags)


def scrape_article(soup, url):
    """Extract one normalized record from a Cornell Chronicle story page."""
    article_json = get_article_json_ld(soup)

    title = clean_text(article_json.get("headline", ""))
    if not title:
        title = first_text(
            soup,
            [
                "main h1",
                "article h1",
                "h1",
            ],
        )

    raw_byline = first_text(
        soup,
        [
            "article .field--name-field-byline",
            "main .field--name-field-byline",
            "article .byline",
            "main .byline",
            "[rel='author']",
        ],
    )
    if not raw_byline:
        raw_byline = names_from_json_ld(article_json.get("author"))

    author, byline_source = split_author_and_source(raw_byline)

    source = first_text(
        soup,
        [
            "article .field--name-field-source",
            "main .field--name-field-source",
            "article .field--name-field-story-source",
            "main .field--name-field-story-source",
            "article .story-source",
            "main .story-source",
        ],
    )
    if not source:
        source = byline_source

    # Publisher is only a final fallback; it may simply be Cornell University.
    if not source:
        source = names_from_json_ld(article_json.get("publisher"))

    published_date = clean_text(article_json.get("datePublished", ""))
    if not published_date:
        published_date = first_attribute(
            soup,
            [
                ('meta[property="article:published_time"]', "content"),
                ('meta[name="date"]', "content"),
                ("time[datetime]", "datetime"),
            ],
        )
    if not published_date:
        published_date = first_text(
            soup,
            [
                "article .field--name-field-date",
                "main .field--name-field-date",
                "article time",
                "main time",
            ],
        )

    sub_heading = first_text(
        soup,
        [
            "article .field--name-field-deck",
            "article .field--name-field-dek",
            "article .field--name-field-summary",
            "main .field--name-field-deck",
            "main .field--name-field-dek",
            "main .field--name-field-summary",
            "article .dek",
            "main .dek",
        ],
    )
    if not sub_heading:
        sub_heading = clean_text(article_json.get("description", ""))
    if not sub_heading:
        sub_heading = first_attribute(
            soup,
            [
                ('meta[name="description"]', "content"),
                ('meta[property="og:description"]', "content"),
            ],
        )

    article_body = extract_body(soup, article_json)
    tags = extract_tags(soup, article_json)

    return {
        "Published Date": published_date,
        "Author": author,
        "Source": source,
        "Article Header": title,
        "Sub_Headings": sub_heading,
        "Article Body": article_body,
        "Tags": tags,
        "Url": url,
    }

In [ ]:
def save_checkpoint(records):
    """Save one deduplicated checkpoint, keeping the newest result per URL."""
    if not records:
        return

    frame = pd.DataFrame(records)
    frame = frame.drop_duplicates(subset="Url", keep="last").reset_index(drop=True)
    frame.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")


# Resume support: retain existing successful or failed rows and skip their URLs.
records = []
completed_urls = set()

if OUTPUT_PATH.exists():
    existing = pd.read_csv(OUTPUT_PATH)
    existing = existing.loc[:, ~existing.columns.str.startswith("Unnamed:")]
    records = existing.to_dict("records")
    if "Url" in existing.columns:
        completed_urls = set(existing["Url"].dropna().astype(str))
    print(f"Resume mode: {len(completed_urls)} URLs already present in {OUTPUT_PATH}.")

In [ ]:
# -----------------------------
# Scrape all article pages
# -----------------------------
pending_rows = links[~links["Link"].astype(str).isin(completed_urls)]

for counter, row in enumerate(
    tqdm(pending_rows.itertuples(index=False), total=len(pending_rows), desc="Articles"),
    start=1,
):
    url = str(row.Link)

    record = {
        "Published Date": "",
        "Author": "",
        "Source": "",
        "Article Header": "",
        "Sub_Headings": "",
        "Article Body": "",
        "Tags": "",
        "Url": url,
        "HTTP Status": "",
        "Error": "",
    }

    try:
        response = session.get(url, timeout=REQUEST_TIMEOUT)
        record["HTTP Status"] = response.status_code
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")
        record.update(scrape_article(soup, url))

        if not record["Article Header"]:
            record["Error"] = "Headline selector returned no value."
        elif not record["Article Body"]:
            record["Error"] = "Article body selector returned no value."

    except requests.RequestException as exc:
        record["Error"] = f"{type(exc).__name__}: {exc}"
    except Exception as exc:
        # Preserve the URL and failure reason instead of terminating the full crawl.
        record["Error"] = f"{type(exc).__name__}: {exc}"

    records.append(record)

    if counter % CHECKPOINT_EVERY == 0:
        save_checkpoint(records)

    sleep(random.uniform(*DELAY_RANGE))

save_checkpoint(records)
print(f"Saved {len(pd.DataFrame(records).drop_duplicates(subset='Url'))} rows to: {OUTPUT_PATH.resolve()}")

In [ ]:
articles = pd.read_csv(OUTPUT_PATH)
articles.head()

In [ ]:
# Inspect failed or incomplete rows before analysis.
problem_rows = articles[
    articles["Error"].fillna("").ne("")
    | articles["Article Header"].fillna("").eq("")
    | articles["Article Body"].fillna("").eq("")
]

problem_rows[["Url", "HTTP Status", "Error"]].head(20)

In [ ]:
# Optional quality checks.
print("Duplicate URLs:", articles["Url"].duplicated().sum())
print("Missing headlines:", articles["Article Header"].isna().sum() + articles["Article Header"].eq("").sum())
print("Missing bodies:", articles["Article Body"].isna().sum() + articles["Article Body"].eq("").sum())
print("Rows with errors:", articles["Error"].fillna("").ne("").sum())